%md
# 01 - Data Exploration: Store Sales Time Series (Corporación Favorita)

%md
## Section 1: Data Loading
Loading the sales table from Unity Catalog into a Spark DataFrame.

In [0]:
df=spark.read.table("workspace.default.sales_train_raw")
df.display()

## Section 2: Volume & Structure
Confirming row count and column types before analysis.

In [0]:
df.count()

In [0]:
print(f"Total raws {df.count()}")

In [0]:
df.printSchema()

## Section 3: Cardinality
Counting distinct stores and product families — this defines the
prediction unit (how many independent time series exist).

In [0]:
df.select("store_nbr").distinct().display()

In [0]:
df.select("family").distinct().display()

**Interpretation:** 54 distinct stores × 33 product families = up to 1,782
possible store-family combinations. This defines the forecasting unit —
we're not predicting a single time series, but potentially up to 1,782
independent ones (store × category level).

## Section 4: Time Range
Checking the earliest and latest dates to confirm enough history
exists to capture yearly seasonality.

In [0]:
from pyspark.sql import functions as F
df.select(F.min("date"), F.max("date")).show()

**Interpretation:** ~4.6 years of daily history. This is enough to capture
at least 4 full annual cycles, which is sufficient to distinguish real
seasonal patterns (e.g., holidays) from random noise.

## Section 5: Target Quality
Checking for negative or zero values in `sales` (the variable we
will forecast) before trusting it as a modeling target.

In [0]:
df.filter(F.col("Sales") < 0).count()

In [0]:
df.filter(F.col("Sales") == 0).count()

**Interpretation:** No negative sales values (0) — the target variable is
clean, no returns/data corruption to handle. Zero sales appear in ~31% of
rows (939,130 / 3,000,888), which is expected given the high cardinality
(54 stores × 33 families = up to 1,782 combinations) — many categories
simply don't sell every day in every store. These zeros are kept as valid
signal, not treated as missing data.

## EDA Summary
- 3,000,888 rows, clean schema
- 54 stores × 33 product families
- ~4.6 years of history (2013-01-01 to 2017-08-15) — enough for seasonality
- Target (`sales`) is clean: no negatives, zeros are valid/expected